# CIFAR-10 CNN EAUT ExperimentNotebook này dùng để chạy lại thực nghiệm từ GitHub trên Google Colab.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi || true

## 2. Clone project từ GitHub
Sau khi đưa project lên GitHub, sửa `REPO_URL` thành link repository.

In [ ]:
REPO_URL = "https://github.com/hoangtnedu/cifar10-cnn-eaut.git"
PROJECT_DIR = "cifar10-cnn-eaut"

import os, shutil
if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
!git clone $REPO_URL

In [ ]:
%cd cifar10-cnn-eaut

!pip install -q -r requirements.txt

## 2.1. Gắn Google Drive để resume bền hơn

Dùng để checkpoint không mất khi Colab reset runtime, hãy mount Google Drive và dùng config `cifar10_5seeds_colab_drive.yaml`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MAIN_CONFIG = 'configs/cifar10_5seeds_colab_drive.yaml'  # lưu checkpoint/kết quả vào Google Drive
# MAIN_CONFIG = 'configs/cifar10_5seeds.yaml'  # dùng nếu không muốn lưu Drive

## 3. Test nhanh bằng config debug



Chạy 1 model, 1 seed, 2 epoch để kiểm tra code trước.

In [ ]:
!python run_experiments.py --config configs/debug.yaml

!python aggregate_results.py --config configs/debug.yaml

## 4. Chạy thực nghiệm chính 5 seed



Lệnh này có thể chạy lâu. Nếu Colab ngắt, chạy lại cell này; project sẽ tự resume từ `checkpoint_last.pt`.

In [ ]:
!python run_experiments.py --config $MAIN_CONFIG

## 5. Tổng hợp kết quả mean ± std

In [ ]:
!python aggregate_results.py --config $MAIN_CONFIG
!cat /content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/aggregate/summary_markdown.md || cat outputs/cifar10_5seeds/aggregate/summary_markdown.md

## 6. Sinh hình và bảng cho bài báo

Cell này đọc kết quả đã huấn luyện và sinh các bảng `.csv`, `.md` cùng các hình `.png` phục vụ đưa vào bài báo. Mặc định script sẽ tạo cả ma trận nhầm lẫn cho seed tốt nhất của từng mô hình. Nếu chỉ muốn sinh nhanh bảng và biểu đồ tổng quan, bỏ comment dòng có `--skip-confusion-matrix`.

In [ ]:
!python make_figures.py --config $MAIN_CONFIG
# Chạy nhanh, không tạo confusion matrix:
# !python make_figures.py --config $MAIN_CONFIG --skip-confusion-matrix

In [ ]:
!echo "=== Figures ==="
!find /content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/figures -maxdepth 1 -type f 2>/dev/null | sort || find outputs/cifar10_5seeds/figures -maxdepth 1 -type f 2>/dev/null | sort

!echo "=== Paper tables ==="
!find /content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/paper_tables -maxdepth 1 -type f 2>/dev/null | sort || find outputs/cifar10_5seeds/paper_tables -maxdepth 1 -type f 2>/dev/null | sort

!echo "=== Main results table ==="
!cat /content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds/paper_tables/table_1_main_results_for_paper.md || cat outputs/cifar10_5seeds/paper_tables/table_1_main_results_for_paper.md

## 6.1. Hiển thị nhanh một số hình chính

In [ ]:
from pathlib import Path
from IPython.display import Image, display

base_dir = Path('/content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds')
if not base_dir.exists():
    base_dir = Path('outputs/cifar10_5seeds')

fig_dir = base_dir / 'figures'
for fig_name in [
    'fig_01_accuracy_f1_bar.png',
    'fig_05_accuracy_vs_params.png',
    'fig_06_accuracy_vs_flops.png',
    'fig_07_accuracy_vs_latency.png',
    'fig_08_pareto_accuracy_latency.png',
    'fig_09_validation_accuracy_curves.png',
]:
    fig_path = fig_dir / fig_name
    if fig_path.exists():
        print(fig_name)
        display(Image(filename=str(fig_path)))
    else:
        print(f'Missing: {fig_path}')

## 7. Nén kết quả để tải về hoặc lưu Google Drive

In [ ]:
!zip -r cifar10_5seeds_outputs.zip /content/drive/MyDrive/cifar10_eaut_outputs/cifar10_5seeds outputs/cifar10_5seeds 2>/dev/null > /dev/null
from google.colab import files
files.download('cifar10_5seeds_outputs.zip')